In [ ]:
# 02 — Feature Engineering
Goal: Downsample to 1-minute, create lag features, and split chronologically.

In [1]:
import pandas as pd
import numpy as np

# 1. Load data
aau = pd.read_csv("../data/raw/AAU_temperature_and_power_use.csv")
aau['Time [Date/Time]'] = pd.to_datetime(aau['Time [Date/Time]'])

# 2. Keep only what we need (to save memory)
# We'll use Temp mod 1 ch 1 as our target, and the 3 power channels
cols_to_keep = [
    'Time [Date/Time]', 
    'Temperature mod 1 ch 1', 
    'Power Ch 1 (W)', 'Power Ch 2 (W)', 'Power Ch 3 (W)'
]
df = aau[cols_to_keep].copy()

# 3. Downsample to 1-minute (averaging)
df = df.set_index('Time [Date/Time]').resample('1min').mean().reset_index()

# 4. Rename columns to be easier to type
df = df.rename(columns={
    'Time [Date/Time]': 'time',
    'Temperature mod 1 ch 1': 'temp',
    'Power Ch 1 (W)': 'power_1',
    'Power Ch 2 (W)': 'power_2',
    'Power Ch 3 (W)': 'power_3'
})

print("Shape after downsampling:", df.shape)
print(df.head())

Shape after downsampling: (22856, 5)
                 time       temp      power_1     power_2     power_3
0 2024-09-16 16:58:00  24.836114  1275.283333  641.675000  431.394444
1 2024-09-16 16:59:00  24.491861  1234.258333  641.968333  431.603333
2 2024-09-16 17:00:00  24.044852  1307.176667  640.921667  431.825000
3 2024-09-16 17:01:00  23.578479  1216.211667  640.935000  431.530000
4 2024-09-16 17:02:00  23.213007  1259.460000  640.650000  431.353333


In [2]:
# 5. Add time features (helps the model learn daily patterns)
df['hour'] = df['time'].dt.hour
df['minute'] = df['time'].dt.minute

# 6. Add the "Regime" feature for the Sept 28 jump
# 0 = before the jump, 1 = after the jump
jump_date = pd.to_datetime('2024-09-28')
df['regime'] = (df['time'] >= jump_date).astype(int)

print(df[['time', 'temp', 'power_1', 'hour', 'regime']].head())

                 time       temp      power_1  hour  regime
0 2024-09-16 16:58:00  24.836114  1275.283333    16       0
1 2024-09-16 16:59:00  24.491861  1234.258333    16       0
2 2024-09-16 17:00:00  24.044852  1307.176667    17       0
3 2024-09-16 17:01:00  23.578479  1216.211667    17       0
4 2024-09-16 17:02:00  23.213007  1259.460000    17       0


In [3]:
# 7. Create Lag Features
# These are the "past" values the model will use to predict the future

# Target: Temp in 15 minutes
df['temp_target_15m'] = df['temp'].shift(-15)

# Past temperatures
df['temp_lag_1m']  = df['temp'].shift(1)
df['temp_lag_5m']  = df['temp'].shift(5)
df['temp_lag_15m'] = df['temp'].shift(15)
df['temp_lag_60m'] = df['temp'].shift(60)

# Past power
df['power_1_lag_1m'] = df['power_1'].shift(1)
df['power_1_lag_5m'] = df['power_1'].shift(5)

# Rolling mean (average of the last 15 minutes)
df['temp_rolling_mean_15m'] = df['temp'].rolling(window=15).mean()

# 8. Drop rows with NaN values (created by shifting and rolling)
df_clean = df.dropna().reset_index(drop=True)

print("Shape after adding lags and dropping NaN:", df_clean.shape)
print(df_clean[['time', 'temp', 'temp_lag_1m', 'temp_target_15m']].head())

Shape after adding lags and dropping NaN: (22781, 16)
                 time       temp  temp_lag_1m  temp_target_15m
0 2024-09-16 17:58:00  24.759401    24.755923        23.518917
1 2024-09-16 17:59:00  24.774772    24.759401        23.213124
2 2024-09-16 18:00:00  24.810821    24.774772        23.368585
3 2024-09-16 18:01:00  24.552326    24.810821        23.813087
4 2024-09-16 18:02:00  24.066739    24.552326        24.216174


In [4]:
# 9. Chronological Split
# Rule: Never shuffle time-series data!
# We train on Sept 16 - Sept 27, and test on Sept 28 - Oct 2

split_date = pd.to_datetime('2024-09-28')

train_df = df_clean[df_clean['time'] < split_date]
test_df  = df_clean[df_clean['time'] >= split_date]

print(f"Train size: {len(train_df)} rows (from {train_df['time'].min()} to {train_df['time'].max()})")
print(f"Test size:  {len(test_df)} rows (from {test_df['time'].min()} to {test_df['time'].max()})")

Train size: 16202 rows (from 2024-09-16 17:58:00 to 2024-09-27 23:59:00)
Test size:  6579 rows (from 2024-09-28 00:00:00 to 2024-10-02 13:38:00)


In [5]:
# 10. Save the clean data for the modeling notebook
train_df.to_csv("../data/processed/train_data.csv", index=False)
test_df.to_csv("../data/processed/test_data.csv", index=False)

print("Saved train and test data to data/processed/")

Saved train and test data to data/processed/
